# Ensemble methods. Exercises


In this section we have only two exercise:

1. Find the best three classifier in the stacking method using the classifiers from scikit-learn package.

2. Build arcing arc-x4 method. 

In [17]:
%store -r data_set
%store -r labels
%store -r test_data_set
%store -r test_labels
%store -r unique_labels

## Exercise 1: Find the best three classifier in the stacking method

Please use the following classifiers:

* Linear regression,
* Nearest Neighbors,
* Linear SVM,
* Decision Tree,
* Naive Bayes,
* QDA.

In [18]:

from sklearn.metrics import accuracy_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

In [19]:
def build_classifiers():

    linear_regression = LinearRegression()
    linear_regression.fit(data_set, labels)

    neighbors = KNeighborsClassifier()
    neighbors.fit(data_set, labels)

    svm = SVC()
    svm.fit(data_set, labels)

    decision_tree = DecisionTreeClassifier()
    decision_tree.fit(data_set, labels)

    naive_bayes = GaussianNB()
    naive_bayes.fit(data_set, labels)

    qda = QuadraticDiscriminantAnalysis()
    qda.fit(data_set, labels)

    return {
        'LinearRegression': linear_regression,
        'KNeighbors':       neighbors,
        'SVM':              svm,
        'DecisionTree':     decision_tree,
        'NaiveBayes':       naive_bayes,
        'QDA':              qda,
    }

In [20]:
def build_stacked_classifier(classifiers):

    train_meta = np.column_stack(
        [clf.predict(data_set) for clf in classifiers]
    )

    stacked_classifier = DecisionTreeClassifier()
    stacked_classifier.fit(train_meta, labels.reshape((130,)))

    test_meta = np.column_stack(
        [clf.predict(test_data_set) for clf in classifiers]
    )

    predicted = stacked_classifier.predict(test_meta)
    return predicted

In [21]:
from itertools import combinations

all_classifiers = build_classifiers()

best_accuracy = -1
best_combo    = None

for combo_names in combinations(all_classifiers.keys(), 3):
    combo_clfs = [all_classifiers[name] for name in combo_names]
    predicted  = build_stacked_classifier(combo_clfs)
    acc        = accuracy_score(test_labels, predicted)
    print(f"{combo_names}  ->  accuracy = {acc:.4f}")
    if acc > best_accuracy:
        best_accuracy = acc
        best_combo    = combo_names

print()
print(f"Best combination : {best_combo}")
print(f"Best accuracy    : {best_accuracy:.4f}")

('LinearRegression', 'KNeighbors', 'SVM')  ->  accuracy = 0.9500
('LinearRegression', 'KNeighbors', 'DecisionTree')  ->  accuracy = 0.9500
('LinearRegression', 'KNeighbors', 'NaiveBayes')  ->  accuracy = 0.9500
('LinearRegression', 'KNeighbors', 'QDA')  ->  accuracy = 0.9500
('LinearRegression', 'SVM', 'DecisionTree')  ->  accuracy = 0.9500
('LinearRegression', 'SVM', 'NaiveBayes')  ->  accuracy = 0.9500
('LinearRegression', 'SVM', 'QDA')  ->  accuracy = 0.9500
('LinearRegression', 'DecisionTree', 'NaiveBayes')  ->  accuracy = 0.9500
('LinearRegression', 'DecisionTree', 'QDA')  ->  accuracy = 0.9500
('LinearRegression', 'NaiveBayes', 'QDA')  ->  accuracy = 0.9500
('KNeighbors', 'SVM', 'DecisionTree')  ->  accuracy = 0.9500
('KNeighbors', 'SVM', 'NaiveBayes')  ->  accuracy = 0.9000
('KNeighbors', 'SVM', 'QDA')  ->  accuracy = 0.9500
('KNeighbors', 'DecisionTree', 'NaiveBayes')  ->  accuracy = 0.9500
('KNeighbors', 'DecisionTree', 'QDA')  ->  accuracy = 0.9500
('KNeighbors', 'NaiveBayes'

## Exercise 2: 

Use the boosting method and change the code to fullfilt the following requirements:

* the weights should be calculated as:
$w_{n}^{(t+1)}=\frac{1+ I(y_{n}\neq h_{t}(x_{n})}{\sum_{i=1}^{N}1+I(y_{n}\neq h_{t}(x_{n})}$,
* the prediction is done with a voting method.

In [22]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier


def generate_data(sample_number, feature_number, label_number):
    data_set = np.random.random_sample((sample_number, feature_number))
    labels   = np.random.choice(label_number, sample_number)
    return data_set, labels

num_labels    = 2
dimension     = 2
test_set_size  = 1000
train_set_size = 5000

train_set, train_labels = generate_data(train_set_size, dimension, num_labels)
test_set,  test_labels  = generate_data(test_set_size,  dimension, num_labels)


number_of_iterations = 10

weights = np.ones(train_set_size) / train_set_size


def train_model(classifier, weights):
    return classifier.fit(X=train_set, y=train_labels, sample_weight=weights)


def calculate_accuracy_vector(predicted, labels):
    return [1 if p == l else 0 for p, l in zip(predicted, labels)]

Fill the two functions below:

In [23]:
def set_new_weights(model):
    predicted = model.predict(train_set)

    indicator = np.array([0 if p == l else 1
                          for p, l in zip(predicted, train_labels)])

    raw_weights = 1 + indicator
    new_weights = raw_weights / raw_weights.sum()
    return new_weights

Train the classifier with the code below:

In [24]:
classifier = DecisionTreeClassifier(max_depth=1, random_state=1)
classifier.fit(X=train_set, y=train_labels)

classifiers = []
for iteration in range(number_of_iterations):
    model   = train_model(classifier, weights)
    weights = set_new_weights(model)
    classifiers.append(model)

print("Final 10 weghts:", weights[:10])

validate_x, validate_label = generate_data(1, dimension, num_labels)

Final weights (first 10): [0.0001333 0.0001333 0.0002666 0.0001333 0.0001333 0.0001333 0.0001333
 0.0001333 0.0002666 0.0002666]


Set the validation data set:

In [25]:
validate_x, validate_label = generate_data(1, dimension, num_labels)

Fill the prediction code:

In [26]:
def get_prediction(x):
    votes = np.array([clf.predict(x)[0] for clf in classifiers])
    counts = np.bincount(votes)
    return np.argmax(counts), votes

Test it:

In [27]:
prediction, votes = get_prediction(validate_x)

print(f"Validate label: {validate_label[0]}")
print(f"Votes: {votes}")
print(f"Prediction     : {prediction}")

Validate label : 1
Votes          : [1 1 1 1 1 1 1 1 1 1]
Prediction     : 1
